# Segment A — First Contact
### Module 1: Claude API & Ecosystem Foundations

**Goal:** make your first API calls, look at exactly what comes back, and watch Claude
fail at something it has no way of knowing — which is the motivation for everything
that follows today.

This notebook is exploratory. We're in a notebook (not a script) because we want to
run one cell, look at the result, tweak, and re-run — that loop is the point.

## 1. Setup

In [ ]:
# If you haven't already, install the SDK:
# pip install anthropic

import os
from anthropic import Anthropic

# The SDK reads ANTHROPIC_API_KEY from the environment automatically.
# Set it in your shell before launching Jupyter, e.g.:
#   export ANTHROPIC_API_KEY="sk-ant-..."
# Or uncomment and set it directly here for class purposes only —
# never commit a real key to a notebook or a repo.

os.environ["ANTHROPIC_API_KEY"] = "sk-ant..."

client = Anthropic()
MODEL = "claude-sonnet-4-6"

print("Client initialized. Using model:", MODEL)

Client initialized. Using model: claude-sonnet-4-6


## 2. Your first call

Let's make the simplest possible request: one user message, no system prompt,
no tools. Just to see the round trip work.

In [2]:
response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    messages=[
        {"role": "user", "content": "In one sentence, what is an AI agent?"}
    ],
)


print(response.model_dump_json(indent=2))

{
  "id": "msg_015yL8jtn5SJvq8PJUg69XsA",
  "container": null,
  "content": [
    {
      "citations": null,
      "text": "An AI agent is an autonomous software system that perceives its environment, makes decisions, and takes actions to achieve specific goals, often using tools or interacting with external systems.",
      "type": "text"
    }
  ],
  "model": "claude-sonnet-4-6",
  "role": "assistant",
  "stop_details": null,
  "stop_reason": "end_turn",
  "stop_sequence": null,
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "global",
    "input_tokens": 17,
    "output_tokens": 38,
    "output_tokens_details": null,
    "server_tool_use": null,
    "service_tier": "standard"
  }
}


That printed the **entire response object**, not just the text. This is deliberate —
in a moment we'll be parsing this object programmatically, so it's worth seeing its
real shape now rather than only ever looking at `.content[0].text`.

Notice:
- `content` is a **list** of blocks, not a single string
- `stop_reason` tells you why generation stopped
- `usage` tells you token counts (this matters for cost — we'll come back to this)

Let's pull out just the text, the way you'll usually want to.

In [3]:
print(response.content[0].text)
print()
print("stop_reason:", response.stop_reason)
print("usage:", response.usage)

An AI agent is an autonomous software system that perceives its environment, makes decisions, and takes actions to achieve specific goals, often using tools or interacting with external systems.

stop_reason: end_turn
usage: Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=17, output_tokens=38, output_tokens_details=None, server_tool_use=None, service_tier='standard')


## 3. Add a system prompt and multi-turn history

The `system` parameter sets persistent behavior. The `messages` list is where the
actual conversation lives — and notice that WE construct this list. There's no
hidden session on Anthropic's servers; if we want Claude to "remember" the system's
behavior, the system string has to be present, and if we want it to remember a
prior turn, that turn has to be in `messages`.

In [4]:
response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    system="You are a terse research assistant. Answer in one or two sentences, no more.",
    messages=[
        {"role": "user", "content": "What is Retrieval-Augmented Generation?"},
    ],
)

print(response.content[0].text)

Retrieval-Augmented Generation (RAG) is a technique that enhances large language model outputs by retrieving relevant documents or data from an external knowledge base at inference time and incorporating that context into the prompt. This allows the model to generate more accurate, up-to-date, and grounded responses without requiring retraining.


Now let's manually continue the conversation by appending to the messages list
ourselves — this is the part that's easy to take for granted once a framework does
it for you, so let's do it by hand once.

In [5]:
messages = [
    {"role": "user", "content": "What is Retrieval-Augmented Generation?"},
    {"role": "assistant", "content": response.content[0].text},
    {"role": "user", "content": "Now explain how that's different from fine-tuning."},
]

response2 = client.messages.create(
    model=MODEL,
    max_tokens=300,
    system="You are a terse research assistant. Answer in one or two sentences, no more.",
    messages=messages,
)

print(response2.content[0].text)

Fine-tuning updates a model's weights by training it on domain-specific data, embedding knowledge directly into the model's parameters. RAG, by contrast, keeps the model's weights frozen and instead supplies relevant information dynamically at query time through retrieval.


Notice we had to manually re-send the **entire** prior exchange, including
Claude's own previous answer, just to ask a follow-up. There's no `client.continue()`
method. That `messages` list is the whole memory, and we are the ones maintaining it.

## 4. Watch Claude fail without tools

Now let's ask something Claude has no way of knowing — something current, specific,
and outside training data. This isn't a trick question; it's the entire motivation
for tool use, which we build in Segment B.

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    messages=[
        {"role": "user", "content": "What is the most recent paper posted to arXiv's cs.AI category today, and who wrote it?"}
    ],
)

print(response.content[0].text)

Run that a couple of times and watch what happens. You'll typically see one of two
failure modes:

1. Claude is upfront that it doesn't have live access to arXiv and can't answer.
2. Claude makes something plausible-sounding up (a hallucinated paper title/author).

Both are the same underlying problem: **the model only has what's in its training
data and what's in this `messages` array.** There is no third source. If the answer
needs the current state of the world, the only way to get it there is to put it in
the array ourselves — which means fetching it from somewhere and inserting it as
content, or (much better) letting Claude ask for it via a tool call.

That's Segment B.

## Recap

- `client.messages.create()` is the entire interface — no sessions, no hidden state
- `response.content` is a list of blocks; `response.content[0].text` is the usual case for plain replies
- `stop_reason` and `usage` are on the response object, not buried in the text
- Conversation "memory" is just us re-sending the growing `messages` list
- Claude can't know things outside training data + what's in the messages array — that's exactly the gap tools fill

Next: Segment B — give Claude tools, and build the loop that lets it use them.